In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import sklearn
from google.colab import drive
import os
import random
from google.colab import files
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Charger les fichiers
interactions = pd.read_csv('https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/interactions_train.csv')
items = pd.read_csv("https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/items.csv")

# Afficher les premières lignes de chaque ensemble de données
display(interactions.head())
display(items.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


In [3]:
# Renommer les colonnes du dataset interactions
interactions.rename(columns={
    'u': 'user_id',        # Colonne utilisateur
    'i': 'item_id',        # Colonne identifiant du livre
    't': 'timestamp'       # Colonne horodatage
}, inplace=True)

# Renommer les colonnes du dataset items
items.rename(columns={
    'Title': 'title',          # Titre du livre
    'Author': 'author',        # Auteur
    'ISBN Valid': 'isbn',      # ISBN
    'Publisher': 'publisher',  # Éditeur
    'Subjects': 'subjects',    # Catégories/thèmes
    'i': 'item_id'             # Identifiant du livre
}, inplace=True)

# Vérifier les colonnes après renommage
print("Colonnes interactions:", interactions.columns)
print("Colonnes items:", items.columns)

Colonnes interactions: Index(['user_id', 'item_id', 'timestamp'], dtype='object')
Colonnes items: Index(['title', 'author', 'isbn', 'publisher', 'subjects', 'item_id'], dtype='object')


In [4]:
#Tri des données par utilisateur et par timestamp
interactions = interactions.sort_values(["user_id", "timestamp"])
print(interactions)

       user_id  item_id     timestamp
21035        0        0  1.680191e+09
28842        0        1  1.680783e+09
3958         0        2  1.680801e+09
29592        0        3  1.683715e+09
6371         0        3  1.683715e+09
...        ...      ...           ...
74604     7836     3471  1.728644e+09
69092     7836     3471  1.728644e+09
51684     7837     2191  1.728735e+09
76172     7837       88  1.728735e+09
30565     7837     2209  1.728735e+09

[87047 rows x 3 columns]


In [5]:
#Calcul du rang proportionnel par utilisateur
interactions["pct_rank"] = interactions.groupby("user_id")["timestamp"].rank(pct=True, method="dense")
interactions.reset_index(inplace=True, drop=True)
print(interactions)

       user_id  item_id     timestamp  pct_rank
0            0        0  1.680191e+09  0.040000
1            0        1  1.680783e+09  0.080000
2            0        2  1.680801e+09  0.120000
3            0        3  1.683715e+09  0.160000
4            0        3  1.683715e+09  0.200000
...        ...      ...           ...       ...
87042     7836     3471  1.728644e+09  0.666667
87043     7836     3471  1.728644e+09  1.000000
87044     7837     2191  1.728735e+09  0.333333
87045     7837       88  1.728735e+09  0.666667
87046     7837     2209  1.728735e+09  1.000000

[87047 rows x 4 columns]


In [6]:
#Calcul de l'atténuation temporelle
time_decay = np.exp(-(interactions["timestamp"].max() - interactions["timestamp"]) / (365 * 24 * 60 * 60))
interactions["time_decay"] = time_decay
print(interactions)

       user_id  item_id     timestamp  pct_rank  time_decay
0            0        0  1.680191e+09  0.040000    0.213238
1            0        1  1.680783e+09  0.080000    0.217280
2            0        2  1.680801e+09  0.120000    0.217405
3            0        3  1.683715e+09  0.160000    0.238448
4            0        3  1.683715e+09  0.200000    0.238448
...        ...      ...           ...       ...         ...
87042     7836     3471  1.728644e+09  0.666667    0.991129
87043     7836     3471  1.728644e+09  1.000000    0.991130
87044     7837     2191  1.728735e+09  0.333333    0.993985
87045     7837       88  1.728735e+09  0.666667    0.993985
87046     7837     2209  1.728735e+09  1.000000    0.993987

[87047 rows x 5 columns]


In [7]:
n_users = interactions["user_id"].max() + 1
n_items = interactions["item_id"].max() + 1
print("Nombre d'utilisateurs:", n_users)
print("Nombre d'items:", n_items)

Nombre d'utilisateurs: 7838
Nombre d'items: 15291


In [8]:
# Définition de la fonction pour créer la matrice utilisateur-item
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["user_id"].values, data["item_id"].values] = data["time_decay"].values
    return data_matrix

# Créer la matrice utilisateur-item avec la pondération temporelle
train_data_matrix = create_data_matrix(interactions, n_users, n_items)
print(train_data_matrix)

# Calcul de la similarité entre items
item_similarity = cosine_similarity(train_data_matrix.T)
print(item_similarity)
print(item_similarity.shape)


# Calcul de la similarité entre utilisateurs
user_similarity = cosine_similarity(train_data_matrix)
print(user_similarity)

# Définition de la fonction pour prédire avec la similarité des items
def item_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

# Générer les prédictions basées sur la similarité des items
item_prediction = item_predict(train_data_matrix, item_similarity)
print(item_prediction)

# Définition de la fonction pour prédire avec la similarité des utilisateurs
def user_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred


# Générer les prédictions basées sur la similarité des utilisateurs
user_prediction = user_predict(train_data_matrix, user_similarity)
print(user_prediction)

[[0.2132384  0.21728029 0.21740485 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]
(15291, 15291)
[[1.         0.         0.         ... 0.         0.         0.        ]
 [0.         1.         0.         ... 0.         0.         0.        ]
 [0.         0.         1.         ... 0.         0.         0.11010201]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.        ]
 [0.         0.

In [9]:
# Affichage ou analyse des résultats (par exemple, visualiser les 5 premières prédictions)
print("Prédictions basées sur les items :")
print(item_prediction[:5, :5])

print("Prédictions basées sur les utilisateurs :")
print(user_prediction[:5, :5])

Prédictions basées sur les items :
[[0.         0.         0.         0.06901787 0.01383286]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.00485433 0.00056062]
 [0.         0.         0.         0.01107654 0.00300526]
 [0.         0.         0.         0.         0.00018039]]
Prédictions basées sur les utilisateurs :
[[0.03207098 0.03138556 0.03352328 0.0358002  0.06732556]
 [0.         0.         0.         0.         0.        ]
 [0.00333777 0.         0.         0.00097367 0.0007886 ]
 [0.00049416 0.         0.         0.00099576 0.00549279]
 [0.         0.         0.         0.         0.00067409]]


In [10]:
# Initialisation du scaler
scaler = MinMaxScaler()

# Normalisation des prédictions
user_prediction_normalized = scaler.fit_transform(user_prediction.reshape(-1, 1)).flatten()
item_prediction_normalized = scaler.fit_transform(item_prediction.reshape(-1, 1)).flatten()

In [11]:
# Définition des poids pour le modèle hybride
x = 0.2 # Poids pour la similarité utilisateur
y = 1 - x # Poids pour la similarité item

model_hybrid = x * user_prediction + y * item_prediction

In [12]:
# Générer les 10 meilleures recommandations avec une logique unifiée
recommendations = []
for user_id in range(model_hybrid.shape[0]):
    top_10 = np.argsort(model_hybrid[user_id, :])[-10:][::-1]
    recommendations.append(" ".join(map(str, top_10)))

# Créer un DataFrame avec les recommandations
import pandas as pd
submission_df = pd.DataFrame({
    "user_id": range(model_hybrid.shape[0]),
    "recommendation": recommendations
})

# Sauvegarder le fichier CSV dans l'environnement Colab
submission_file = "predictions_submission.csv"
submission_df.to_csv(submission_file, index=False)

# Télécharger le fichier localement
from google.colab import files
files.download(submission_file)

# Vérifier le résultat
print(submission_df.head(10))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   user_id                                     recommendation
0        0                      21 24 20 17 7 6 22 8664 14 11
1        1                   31 35 39 33 13434 32 38 36 37 29
2        2                      94 80 92 76 50 57 79 77 60 82
3        3            157 132 145 140 151 162 155 134 168 166
4        4            192 204 203 202 200 205 195 194 197 196
5        5            221 212 223 222 219 224 218 217 215 213
6        6    229 232 230 231 2829 5638 7568 13988 3862 13406
7        7            246 240 250 236 245 247 249 243 242 239
8        8  258 257 256 14986 13371 14417 4385 14483 5920 ...
9        9    262 264 261 263 1556 13709 4381 11774 7954 1524
